# 4. Planning, model routing e subagenti

Costruiamo un supervisor LangChain che coordina due agenti specializzati:

- un **analista**, autorizzato a usare un tool statistico;
- un **revisore**, isolato dal contesto operativo e incaricato di criticare la bozza.

Il supervisor usa `TodoListMiddleware` per pianificare e un middleware personalizzato per scegliere tra modello economico e modello forte.

## Configurazione dei modelli

In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI


def find_env() -> Path:
    current = Path.cwd().resolve()
    for directory in (current, *current.parents):
        candidate = directory / '.env'
        if candidate.is_file():
            return candidate
    raise FileNotFoundError('.env non trovato.')


ENV_FILE = find_env()
load_dotenv(ENV_FILE, override=False)
if not os.getenv('OPENAI_API_KEY'):
    raise RuntimeError(f'OPENAI_API_KEY non configurata in {ENV_FILE}')

FAST_MODEL_NAME = os.getenv('OPENAI_MODEL') or 'gpt-5.4-mini'
STRONG_MODEL_NAME = os.getenv('OPENAI_STRONG_MODEL') or FAST_MODEL_NAME
fast_model = ChatOpenAI(
    model=FAST_MODEL_NAME,
    reasoning_effort='low',
    use_responses_api=True,
    store=False,
)
strong_model = ChatOpenAI(
    model=STRONG_MODEL_NAME,
    reasoning_effort='medium',
    use_responses_api=True,
    store=False,
)
print('Modello veloce:', FAST_MODEL_NAME)
print('Modello forte:', STRONG_MODEL_NAME)

## Subagente analista

L'analista possiede un tool deterministico. Il supervisor non vede i messaggi intermedi dell'analista: riceve soltanto il suo ultimo messaggio.

In [ ]:
import statistics

from langchain.agents import create_agent
from langchain.tools import tool


@tool
def descriptive_statistics(values: list[float]) -> dict[str, float]:
    '''Calcola conteggio, minimo, massimo, media e mediana di una lista numerica non vuota.'''
    if not values:
        raise ValueError('La lista non può essere vuota.')
    return {
        'count': float(len(values)),
        'minimum': min(values),
        'maximum': max(values),
        'mean': statistics.fmean(values),
        'median': statistics.median(values),
    }


analyst = create_agent(
    model=fast_model,
    tools=[descriptive_statistics],
    system_prompt=(
        'Sei un analista quantitativo. Usa sempre descriptive_statistics per i numeri. '
        'Restituisci risultati, interpretazione e limiti in massimo 180 parole.'
    ),
)


@tool
def delegate_analysis(task: str) -> str:
    '''Delega al subagente analista un compito quantitativo e restituisce soltanto la sua sintesi finale.'''
    result = analyst.invoke({'messages': [{'role': 'user', 'content': task}]})
    return result['messages'][-1].text

## Subagente revisore

Il revisore usa il modello forte ma nessun tool. Riceve soltanto la bozza selezionata dal supervisor: questa è **context quarantine**, non semplice chiamata a una seconda persona virtuale.

In [ ]:
reviewer = create_agent(
    model=strong_model,
    tools=[],
    system_prompt=(
        'Sei un revisore indipendente. Controlla coerenza numerica, inferenze non supportate, '
        'limiti del campione e chiarezza. Restituisci correzioni concrete, non una nuova analisi.'
    ),
)


@tool
def delegate_review(draft: str) -> str:
    '''Invia una bozza al subagente revisore e restituisce soltanto il rapporto finale.'''
    result = reviewer.invoke({'messages': [{'role': 'user', 'content': draft}]})
    return result['messages'][-1].text

## Middleware di model routing

`wrap_model_call` intercetta ogni chiamata del supervisor. La policy è osservabile: richieste di revisione finale o contesti lunghi usano il modello forte; il resto usa quello economico.

In [ ]:
from collections.abc import Callable

from langchain.agents.middleware import ModelRequest, ModelResponse, wrap_model_call

routing_log: list[str] = []


@wrap_model_call
def route_model(
    request: ModelRequest,
    handler: Callable[[ModelRequest], ModelResponse],
) -> ModelResponse:
    last_text = str(request.messages[-1].content).casefold() if request.messages else ''
    use_strong = len(request.messages) > 12 or 'revisione finale' in last_text
    selected = strong_model if use_strong else fast_model
    routing_log.append(STRONG_MODEL_NAME if use_strong else FAST_MODEL_NAME)
    return handler(request.override(model=selected))

## Supervisor con piano obbligatorio

`TodoListMiddleware` aggiunge il tool `write_todos`. Il prompt descrive l'ordine delle dipendenze: prima analisi, poi bozza, poi revisione, infine risposta corretta.

In [ ]:
from langchain.agents.middleware import TodoListMiddleware

supervisor = create_agent(
    model=fast_model,
    tools=[delegate_analysis, delegate_review],
    middleware=[TodoListMiddleware(), route_model],
    system_prompt=(
        'Sei il supervisor. Per questa richiesta devi: '
        '(1) creare un piano con write_todos; '
        '(2) delegare i calcoli a delegate_analysis; '
        '(3) preparare una bozza; '
        '(4) inviare la bozza a delegate_review; '
        '(5) incorporare le correzioni e chiudere i todo. '
        'Non saltare delegazioni e non rifare mentalmente i calcoli.'
    ),
)

supervisor_result = supervisor.invoke({
    'messages': [{
        'role': 'user',
        'content': (
            'Analizza i tempi di completamento [12, 18, 21, 21, 28, 30] minuti. '
            'Prepara una raccomandazione operativa prudente e sottoponila a revisione finale.'
        ),
    }]
})
print(supervisor_result['messages'][-1].text)

## Verificare orchestrazione e isolamento

In [ ]:
called_tools = [
    call['name']
    for message in supervisor_result['messages']
    for call in (getattr(message, 'tool_calls', None) or [])
]
print('Tool del supervisor:', called_tools)
print('Decisioni del router:', routing_log)

assert 'write_todos' in called_tools
assert 'delegate_analysis' in called_tools
assert 'delegate_review' in called_tools
assert routing_log

# Il supervisor conserva soltanto i ToolMessage finali delle delegazioni,
# non la cronologia interna dei due subagenti.
delegate_messages = [
    message for message in supervisor_result['messages']
    if type(message).__name__ == 'ToolMessage'
    and getattr(message, 'name', '') in {'delegate_analysis', 'delegate_review'}
]
print('Risultati delegati visibili al supervisor:', len(delegate_messages))

## Quando usare questo pattern

I subagenti hanno senso quando servono tool, prompt, modelli o contesti diversi. Per una domanda semplice aggiungono solo latenza e costo. Il supervisor mantiene controllo centrale; i worker restano stateless e restituiscono una sintesi delimitata.